# Session 7 — Bellman, dynamic programming, Monte Carlo, TD

**Salih Tutun, PhD** · DAT 5668 · Module 4 · Concept session (about 80 minutes)**

I am Salih Tutun. I talk. You guess to a neighbor. Until I say **open the notebook**, this file stays closed. **Do not scroll.** One cell at a time. Two voices, then type. After we run: **LLM**, **RL**, or **hybrid** — and one line for your project.

**Family this session: RL-based agent.** The decide step is a value, not a tool call. The LLM-based agent from Sessions 1 through 3 is the other family. It comes back in Session 11 as the exception handler.

![Bellman backup](../../assets/bellman.png)

## Run of show
- 10 min — Recap the inventory MDP from Session 5.
- 20 min — Value, the Bellman equation, and credit over time.
- 35 min — Demo: one-step backup, value iteration on the known model, then one Monte Carlo return and one TD update.
- 15 min — Why dynamic programming dies on real operations data. Session 8 trains Q-learning, which does not need the model.

## Learning goals
- Read a value as "expected discounted reward from this state if we keep acting this way."
- Compute a Bellman backup when demand probabilities are known.
- Contrast Monte Carlo (wait for the end) with temporal difference (update from the next step).


## How we work in this notebook

I am Salih. I talk. You guess to a neighbor. Two voices share. Then we type **this cell only**. Do not scroll.

| You will see | What you do in class |
| --- | --- |
| **STOP** | Wait. Do not run the next cell until I say. |
| **This cell — ...** | I tell you what we are looking at. Then we run it together. |
| **YOU FILL THIS** | Guess to a neighbor (20 seconds). Two people share. Then type `YOU TYPE HERE`. Then run. |
| **RUN WITH ME** | Already written. Run it. Do not edit. |
| **Family check** | Say **LLM**, **RL**, or **hybrid** out loud — and one line you will keep for your project. |

There is no `TODO` in a concept session. Practice sessions (even numbers) are where you write the function.


In [ ]:
import numpy as np

from pathlib import Path
from collections import Counter, defaultdict
import csv

def course_data(name):
    for base in (Path("data"), Path("../../assets/data"), Path("assets/data"), Path("/content/data")):
        path = base / name
        if path.exists():
            return path
    raise FileNotFoundError(name)


def empirical_demand_p():
    """Bin daily Home-and-lifestyle quantity into {0, 1, 2}."""
    daily = defaultdict(int)
    with course_data("supermarket-sales.csv").open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            if (row.get("Product line") or "").strip() != "Home and lifestyle":
                continue
            daily[row["Date"]] += int(float(row["Quantity"]))
    totals = sorted(daily.values())
    q1, q2 = totals[len(totals) // 3], totals[(2 * len(totals)) // 3]
    bins = [0 if qty <= q1 else 1 if qty <= q2 else 2 for qty in totals]
    counts = Counter(bins)
    n = max(len(bins), 1)
    return {level: counts[level] / n for level in (0, 1, 2)}


DEMAND_P = empirical_demand_p()
DEMAND_LEVELS = list(DEMAND_P.keys())
DEMAND_PROBS = [DEMAND_P[k] for k in DEMAND_LEVELS]
print("Empirical P(demand) from Home-and-lifestyle invoices:", {k: round(v, 3) for k, v in DEMAND_P.items()})

CAPACITY = 4
GAMMA = 0.95
HOLD, MARGIN, STOCKOUT = 0.2, 2.0, 1.5


def transition(stock, order, demand):
    started = min(CAPACITY, stock + order)
    sales = min(started, demand)
    ending = started - sales
    missed = max(0, demand - started)
    pay = MARGIN * sales - HOLD * ending - STOCKOUT * missed
    return ending, pay


def bellman_backup(values, stock, order):
    """Expected reward plus discounted value of the next stock. Model is known. P(demand) is empirical."""
    total = 0.0
    for demand, prob in DEMAND_P.items():
        ending, pay = transition(stock, order, demand)
        total += prob * (pay + GAMMA * values[ending])
    return total


# Start from zeros and apply the best backup to every state. This is value iteration.
values = np.zeros(CAPACITY + 1)
for sweep in range(40):
    new_values = np.zeros_like(values)
    for stock in range(CAPACITY + 1):
        new_values[stock] = max(bellman_backup(values, stock, order) for order in range(3))
    values = new_values

print("Value of each stock level under the best order (model known):")
for stock, value in enumerate(values):
    best_order = int(np.argmax([bellman_backup(values, stock, order) for order in range(3)]))
    print(f"  stock {stock}: V={value:.2f}  best order={best_order}")


## Monte Carlo vs TD, in one episode

Dynamic programming used P(demand). Often you do not have that table. You have days of experience.

- **Monte Carlo** waits until a short episode ends, then assigns the same return to the states you visited. Credit arrives late.
- **Temporal difference (TD)** updates after one step, using the next state's current value as a stand-in for the rest of the future. Credit arrives immediately and is biased until the values settle.

Session 8's Q-learning is a TD method for **state-action** values, and it does not need `P_DEMAND`.


In [ ]:
# One episode under "order 1 whenever stock is below 2".
def behavior_order(stock):
    return 1 if stock < 2 else 0


episode = []
stock = 1
# Fixed demands so the board matches the notebook if you rerun it.
for demand in (2, 0, 1, 2):
    order = behavior_order(stock)
    ending, pay = transition(stock, order, demand)
    episode.append((stock, pay))
    stock = ending

# Monte Carlo return, walking backward. G_t = r_t + gamma * G_{t+1}
monte_carlo = []
return_from_here = 0.0
for stock, pay in reversed(episode):
    return_from_here = pay + GAMMA * return_from_here
    monte_carlo.append((stock, return_from_here))
monte_carlo.reverse()
print("Monte Carlo targets (stock, return from that day):")
for stock, target in monte_carlo:
    print(f"  stock {stock}: {target:.2f}")

# One TD(0) update on the first step, toward r + gamma * V(next).
# Use the value function from dynamic programming as the current guess.
stock0, pay0 = episode[0]
# Reconstruct the next stock from the second visit if present, else from the transition.
next_stock = episode[1][0]
td_target = pay0 + GAMMA * values[next_stock]
print(f"TD target for stock {stock0}: {td_target:.2f}  (current V={values[stock0]:.2f})")
print("The gap is what a learning rule would shrink.")


## When dynamic programming is the wrong tool

You cannot run the backup above if demand probabilities are unknown, the state includes every SKU in a warehouse, or competitors change the transition. That is normal in operations. Model-free learning (Session 8) trades that table for **samples**, and samples cost money because the agent tries bad orders.

Say this to managers: DP is the clean answer on a known small model. It is a whiteboard, not a warehouse system, once the state gets large.


## Fill this with me — type this in class

**STOP. Do not scroll.** Neighbor 20s. Two voices. Look at the value-iteration table. What order does it place when the shelf is empty? Type 0, 1, or 2.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor first. 0, 1, or 2.
order_at_empty = None  # YOU TYPE HERE
learned = int(np.argmax([bellman_backup(values, 0, order) for order in range(3)]))
print("You said", order_at_empty, "| value iteration says", learned)
print("If those differ, tell the room why you would order something else.")


## Add this to your framework — STOP. Do not scroll.

Guess to a neighbor (20 seconds): after today, is **your** project **LLM**, **RL**, or **hybrid**? Two voices share. Then type one sentence you will keep.

Hint: value iteration owns the **numeric** order (RL). Emails and damaged-item notes are not in this table — that is still the LLM family, or a person.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor first. Two voices. Then type.
framework_family = ""  # YOU TYPE HERE: LLM, RL, or hybrid
framework_line = ""    # YOU TYPE HERE: one sentence you will keep
print("Family:", framework_family or "(say it, then type it)")
print("I will keep:", framework_line or "(one sentence)")
